In [2]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers pypdf requests


In [1]:
import os
from pathlib import Path
BASE_PATH = r"D:\\CODE\\Alzheimer_Detection_And_Classification\\rag"
PDF_PATH = os.path.join(BASE_PATH, "data", "Treatment_doc")
VECTOR_PATH = os.path.join(BASE_PATH, "vector_db")
print('BASE_PATH =', BASE_PATH)
print('PDF_PATH =', PDF_PATH)
print('VECTOR_PATH =', VECTOR_PATH)
for p in [BASE_PATH, PDF_PATH, VECTOR_PATH]:
 print(f'Exists? {p} ->', os.path.exists(p))


BASE_PATH = D:\\CODE\\Alzheimer_Detection_And_Classification\\rag
PDF_PATH = D:\\CODE\\Alzheimer_Detection_And_Classification\\rag\data\Treatment_doc
VECTOR_PATH = D:\\CODE\\Alzheimer_Detection_And_Classification\\rag\vector_db
Exists? D:\\CODE\\Alzheimer_Detection_And_Classification\\rag -> True
Exists? D:\\CODE\\Alzheimer_Detection_And_Classification\\rag\data\Treatment_doc -> True
Exists? D:\\CODE\\Alzheimer_Detection_And_Classification\\rag\vector_db -> True


In [3]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

pdf_files = list(Path(PDF_PATH).glob('*.pdf'))
print('PDF count =', len(pdf_files))
for f in pdf_files[:10]:
    print('-', f.name)

if not pdf_files:
    raise FileNotFoundError(f'No PDFs found in: {PDF_PATH}')


C:\Users\arnab\anaconda3\envs\p_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF count = 4
- A Guide to Caring for a Person with Alzheimer's Disease (1).pdf
- ADEARUnravelingtheMystery.pdf
- alzheimers.pdf
- the_dementia_guide.pdf


In [4]:
loader = PyPDFDirectoryLoader(PDF_PATH, glob='**/*.pdf')
docs = loader.load()
print('Loaded pages/documents =', len(docs))

print('\nSample metadata:')
print(docs[0].metadata if docs else 'No docs')

print('\nSample text preview:')
print(docs[0].page_content[:1200] if docs else 'No content')


Loaded pages/documents = 447

Sample metadata:
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-09-16T15:51:17-04:00', 'author': 'National Institute on Aging', 'keywords': "caregiving; Alzheimer's disease; dementia; stages of Alzheimer's disease; activities of daily living; caregiver support", 'moddate': '2013-10-08T14:13:06-04:00', 'title': 'Caring for a Person with Alzheimer’s Disease: Your Easy-to-Use Guide from the National Institute on Aging', 'trapped': '/Unknown', 'source': "D:\\CODE\\Alzheimer_Detection_And_Classification\\rag\\data\\Treatment_doc\\A Guide to Caring for a Person with Alzheimer's Disease (1).pdf", 'total_pages': 108, 'page': 0, 'page_label': '1'}

Sample text preview:
Caring
for a Person with
Alzheimer’s Disease
Your Easy-to-Use Guide  
from the National Institute on Aging


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=180,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks = splitter.split_documents(docs)

for idx, chunk in enumerate(chunks):
    chunk.metadata['chunk_id'] = f'chunk-{idx+1}'

print('Total chunks =', len(chunks))
print('First chunk metadata =', chunks[0].metadata if chunks else 'No chunks')
print('First chunk preview =\n', chunks[0].page_content[:1000] if chunks else 'No chunks')


Total chunks = 1408
First chunk metadata = {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-09-16T15:51:17-04:00', 'author': 'National Institute on Aging', 'keywords': "caregiving; Alzheimer's disease; dementia; stages of Alzheimer's disease; activities of daily living; caregiver support", 'moddate': '2013-10-08T14:13:06-04:00', 'title': 'Caring for a Person with Alzheimer’s Disease: Your Easy-to-Use Guide from the National Institute on Aging', 'trapped': '/Unknown', 'source': "D:\\CODE\\Alzheimer_Detection_And_Classification\\rag\\data\\Treatment_doc\\A Guide to Caring for a Person with Alzheimer's Disease (1).pdf", 'total_pages': 108, 'page': 0, 'page_label': '1', 'chunk_id': 'chunk-1'}
First chunk preview =
 Caring
for a Person with
Alzheimer’s Disease
Your Easy-to-Use Guide  
from the National Institute on Aging


In [7]:
embedding_model = 'sentence-transformers/all-MiniLM-L6-v2'
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vec = embeddings.embed_query('Alzheimer moderate stage treatment and caregiver support')
print('Embedding length =', len(vec))
print('First 10 values =', vec[:10])


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4874.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding length = 384
First 10 values = [-0.05529186129570007, -0.022604811936616898, 0.014754296280443668, 0.010782831348478794, -0.08791100978851318, 0.07367318123579025, -0.025847453624010086, 0.023095006123185158, -0.01017898041754961, -0.06374163925647736]


In [8]:
vector_store = Chroma(
    collection_name='alzheimer_treatment_docs',
    embedding_function=embeddings,
    persist_directory=VECTOR_PATH,
)

try:
    vector_store.reset_collection()
except Exception as e:
    print('reset_collection warning:', e)

vector_store.add_documents(chunks)
print('Vector DB created at:', VECTOR_PATH)


Vector DB created at: D:\\CODE\\Alzheimer_Detection_And_Classification\\rag\vector_db


In [9]:
query = 'Patient is in moderate stage Alzheimer. What should be done now? Give medicinal treatment and non-medicinal treatment.'
results = vector_store.similarity_search(query, k=4)

print('Retrieved documents =', len(results))
for i, doc in enumerate(results, 1):
    print('\n' + '='*80)
    print(f'Result {i}')
    print('Source :', doc.metadata.get('source'))
    print('Page   :', doc.metadata.get('page'))
    print('Chunk  :', doc.metadata.get('chunk_id'))
    print('Text   :')
    print(doc.page_content[:1500])


Retrieved documents = 4

Result 1
Source : D:\CODE\Alzheimer_Detection_And_Classification\rag\data\Treatment_doc\A Guide to Caring for a Person with Alzheimer's Disease (1).pdf
Page   : 0
Chunk  : chunk-1
Text   :
Caring
for a Person with
Alzheimer’s Disease
Your Easy-to-Use Guide  
from the National Institute on Aging

Result 2
Source : D:\CODE\Alzheimer_Detection_And_Classification\rag\data\Treatment_doc\alzheimers.pdf
Page   : 12
Chunk  : chunk-496
Text   :
Practice Guideline for the Treatment of Patients With Alzheimer’s Disease and Other Dementias 13
 
possible and safe, such underlying causes should be
treated ﬁrst [I]. If this does not resolve the symptoms, and
if they do not cause signiﬁcant danger or distress to the
patient or others, such symptoms are best treated with en-
vironmental measures, including reassurance and redirec-
tion [I]. For agitation, some of the behavioral measures
discussed in Section I.B.2 may also be helpful [II]. If these
measures are unsuccessful or t

In [10]:
stage = 'moderate'
user_query = 'what to do now, medicinal treatment, and non-medicinal treatment'
combined_query = f'Alzheimer stage: {stage}. Question: {user_query}'
results = vector_store.similarity_search(combined_query, k=4)

context = []
for doc in results:
    context.append(
        f"Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')} | Chunk: {doc.metadata.get('chunk_id')}\n{doc.page_content}"
    )

final_context = '\n\n'.join(context)
print(final_context[:5000])


Source: D:\CODE\Alzheimer_Detection_And_Classification\rag\data\Treatment_doc\the_dementia_guide.pdf | Page: 45 | Chunk: chunk-1172
The dementia guide
44
Non-drug treatments
Drugs aren’t the only way to treat or manage the 
symptoms of dementia. There are many other things 
that can help you to live well. Some common non-drug 
treatments are mentioned in this chapter and some 
other approaches to living well are covered in ‘Living well’ 
starting on page 50.
The non-drug treatments that are available, and how to 
get them, can vary depending on where you live. Ask your 
GP , memory service or Alzheimer’s Society for details.
You may be able to apply for or refer yourself to some of 
the services mentioned.

Source: D:\CODE\Alzheimer_Detection_And_Classification\rag\data\Treatment_doc\the_dementia_guide.pdf | Page: 35 | Chunk: chunk-1157
The dementia guide
34
There is no known cure for dementia yet. Treatment 
includes drug and non-drug approaches, looking 
after other medical condition